# Fine-tune Mistral-7b with SFT

> 🗣️ [Large Language Model Course](https://github.com/mlabonne/llm-course)

❤️ Created by [@maximelabonne](https://twitter.com/maximelabonne).

You can run this notebook on a free-tier Google Colab (T4 GPU).

In [1]:
!pip install -qqq -U transformers datasets accelerate peft trl bitsandbytes wandb --progress-bar off

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.3.2 requires fsspec==2025.3.2, but you have fsspec 2025.3.0 which is incompatible.


In [2]:
import peft

In [ ]:


import gc
import os
import pickle
import torch
import wandb
from datasets import load_dataset
from google.colab import userdata
from peft import LoraConfig, PeftModel, prepare_model_for_kbit_training
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments,
    pipeline,
)
from trl import SFTTrainer

# Model
base_model = "alpindale/Mistral-7B-v0.2-hf"
new_model = "mistral-7b-miniplatypus"

# Defined in the secrets tab in Google Colab
# wb_token = userdata.get('wandb')
# wandb.login(key=wb_token)

# Set torch dtype and attention implementation
if torch.cuda.get_device_capability()[0] >= 8:
    !pip install -qqq flash-attn
    torch_dtype = torch.bfloat16
    attn_implementation = "flash_attention_2"
else:
    torch_dtype = torch.float16
    attn_implementation = "eager"

## Fine-tuning Mistral-7b

In [ ]:
documents = pickle.load(open('/content/results_checkpoint (1).pkl', 'rb'))
import datasets
import pandas as pd

# Convert the list to a pandas DataFrame first
df = pd.DataFrame(documents['results'])

# Create the dataset from the DataFrame
dataset = datasets.Dataset.from_pandas(df)
# dataset
# suprimer les features "Question" et Answer
dataset = dataset.remove_columns(['Question', 'Answer'])

dataset = dataset.train_test_split(test_size=0.2)
dataset

DatasetDict({
    train: Dataset({
        features: ['question', 'answer'],
        num_rows: 160
    })
    test: Dataset({
        features: ['question', 'answer'],
        num_rows: 40
    })
})

In [ ]:
# Insert your dataset here
dataset_name = "mlabonne/mini-platypus"
dataset = load_dataset(dataset_name, split="all")

README.md:   0%|          | 0.00/316 [00:00<?, ?B/s]

train-00000-of-00001.parquet:   0%|          | 0.00/2.25M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1000 [00:00<?, ? examples/s]

In [ ]:
dataset

Dataset({
    features: ['instruction', 'output'],
    num_rows: 1000
})

In [ ]:



# QLoRA config
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

# LoRA config
peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=['up_proj', 'down_proj', 'gate_proj', 'k_proj', 'q_proj', 'v_proj', 'o_proj']
)

# Tokenizer
tokenizer = AutoTokenizer.from_pretrained(base_model)
tokenizer.pad_token = tokenizer.unk_token

# Load model
model = AutoModelForCausalLM.from_pretrained(
    base_model,
    quantization_config=bnb_config,
    device_map="auto",
    attn_implementation=attn_implementation
)
model = prepare_model_for_kbit_training(model)

tokenizer_config.json:   0%|          | 0.00/960 [00:00<?, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.80M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

...

model-00003-of-00003.safetensors:   0%|          | 0.00/4.54G [00:00<?, ?B/s]

model-00001-of-00003.safetensors:   0%|          | 0.00/4.94G [00:00<?, ?B/s]

model-00002-of-00003.safetensors:   0%|          | 0.00/5.00G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

In [ ]:
dataset

DatasetDict({
    train: Dataset({
        features: ['question', 'answer'],
        num_rows: 160
    })
    test: Dataset({
        features: ['question', 'answer'],
        num_rows: 40
    })
})

In [ ]:
dataset['train'][7]['output']

'To derive the formula for the area of a trapezoid using calculus, we\'ll first define the trapezoid and set up an integral to represent its area.\n\n1. Define the trapezoid:\nA trapezoid is a quadrilateral with one pair of parallel sides. Let\'s call these parallel sides "bases" and label them b1 and b2, where b1 is the longer base and b2 is the shorter base. The distance between these bases (the height) is h. \n\n2. Set up a coordinate system:\nPlace the trapezoid in the xy-plane such that the longer base b1 lies on the x-axis from x = 0 to x = b1, and the shorter base b2 is parallel to the x-axis at y = h. Now, let\'s find the equations of the non-parallel sides (called legs).\n\n3. Find the equations of the legs:\nThe left leg connects the points (0, 0) and (a, h), where a is the horizontal distance from the origin to the point where the shorter base intersects the left leg. The equation of this line can be found using the slope-intercept form: y = mx + c.\nSince it passes through 

In [ ]:
from transformers import TrainingArguments
from trl import SFTTrainer

# 1. Correction des arguments de TrainingArguments
training_arguments = TrainingArguments(
    learning_rate=2e-4,
    lr_scheduler_type="linear",
    num_train_epochs=1,
    per_device_train_batch_size=10,
    per_device_eval_batch_size=10,
    gradient_accumulation_steps=1,
    eval_strategy="steps",      # Certaines versions utilisent encore 'evaluation_strategy'
    eval_steps=0.2,                   # Valide si 0.2 est un ratio acceptable
    logging_steps=1,
    optim="paged_adamw_8bit",
    warmup_steps=10,
    output_dir="./results",
)

# 2. Fonction de formatage des données
def formatting_func(example):
    return f"### Instruction: {example['question']}\n### Response: {example['answer']}"

# 3. Configuration du tokenizer (avant de créer le trainer)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

trainer = SFTTrainer(
    model=model,
    args=training_arguments,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
    peft_config=peft_config,
    formatting_func=formatting_func,
    # max_seq_length=512,
    # Le tokenizer est automatiquement récupéré depuis le modèle
    # Si nécessaire, utilisez 'processing_class=tokenizer' selon votre version de trl
    # packing=True,
)

trainer.train()

trainer.model.save_pretrained(new_model)
tokenizer.save_pretrained(new_model)

/usr/local/lib/python3.11/dist-packages/peft/mapping_func.py:73: UserWarning: You are trying to modify a model with PEFT for a second time. If you want to reload the model with a different config, make sure to call `.unload()` before.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/peft/tuners/tuners_utils.py:167: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


Applying formatting function to train dataset:   0%|          | 0/160 [00:00<?, ? examples/s]

Converting train dataset to ChatML:   0%|          | 0/160 [00:00<?, ? examples/s]

Adding EOS to train dataset:   0%|          | 0/160 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/160 [00:00<?, ? examples/s]

...

 ··········


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: osman77 (esp-sn) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.
/usr/local/lib/python3.11/dist-packages/torch/_dynamo/eval_frame.py:745: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss,Validation Loss
4,3.066800,2.453126
8,2.242600,2.041704


OutOfMemoryError: CUDA out of memory. Tried to allocate 560.00 MiB. GPU 0 has a total capacity of 14.74 GiB of which 2.12 MiB is free. Process 8273 has 14.67 GiB memory in use. Of the allocated memory 12.63 GiB is allocated by PyTorch, and 1.91 GiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [ ]:
# 61925db1bd1f51162c43a30e0986f84ccc446300

In [ ]:
# !pip install -qqq -U trl --progress-bar off # Upgrade trl to the latest version

In [ ]:
# print(help(SFTTrainer))

In [ ]:
prompt = "What is a large language model?"
instruction = f"### Instruction:\n{prompt}\n\n### Response:\n"

pipe = pipeline("text-generation", model=model, tokenizer=tokenizer, max_length=128)
result = pipe(instruction)
print(result[0]["generated_text"][len(instruction):])

NameError: name 'pipeline' is not defined

Merging the base model with the trained adapter.

In [ ]:
# Flush memory
del trainer, model
gc.collect()
torch.cuda.empty_cache()

model = AutoModelForCausalLM.from_pretrained(
    base_model,
    low_cpu_mem_usage=True,
    return_dict=True,
    torch_dtype=torch.float16,
    device_map={"": 0},
)
model = PeftModel.from_pretrained(model, new_model)
model = model.merge_and_unload()

# Reload tokenizer to save it
tokenizer = AutoTokenizer.from_pretrained(base_model, trust_remote_code=True)
tokenizer.pad_token = tokenizer.unk_token

NameError: name 'trainer' is not defined

Optional: pushing the model and tokenizer to the Hugging Face Hub.

In [ ]:
model.push_to_hub(new_model, use_temp_dir=False)
tokenizer.push_to_hub(new_model, use_temp_dir=False)

## Going further

* **DPO fine-tuning**: see [this article](https://mlabonne.github.io/blog/posts/Fine_tune_Mistral_7b_with_DPO.html)
* **Better fine-tuning tool**: see [Axolotl](https://mlabonne.github.io/blog/posts/A_Beginners_Guide_to_LLM_Finetuning.html)
* **Evaluation**: see the [LM Evaluation Harness](https://github.com/EleutherAI/lm-evaluation-harness) and the [Open LLM Leaderboard](https://huggingface.co/spaces/HuggingFaceH4/open_llm_leaderboard)
* **Quantization**: see [naive quantization](https://mlabonne.github.io/blog/posts/Introduction_to_Weight_Quantization.html), [GPTQ](https://mlabonne.github.io/blog/posts/4_bit_Quantization_with_GPTQ.html), [GGUF/llama.cpp](https://mlabonne.github.io/blog/posts/Quantize_Llama_2_models_using_ggml.html), ExLlamav2, and AWQ.